# Projeto (Seção 3) — solução comentada

Solução própria, testes extras (o enunciado pede explicitamente), revisão da sua tentativa (`10 - My_Project` e `10 - My_Project.py`)
e comparação com o instrutor. Enunciado em [`10_projeto`](10_projeto.ipynb).

## 1. Solução

In [1]:
from datetime import date
from enum import Enum
from pydantic import BaseModel, ConfigDict, ValidationError


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",               # campo desconhecido -> erro
        str_strip_whitespace=True,    # " BMW " -> "BMW" (também em atribuições)
        validate_default=True,        # defaults passam pela validação
        validate_assignment=True,     # car.campo = valor é validado
    )

    manufacturer: str
    series_name: str
    type_: AutomobileType
    is_electric: bool = False
    manufactured_date: date
    base_msrp_usd: float
    vin: str
    number_of_doors: int = 4
    registration_country: str | None = None
    license_plate: str | None = None

## 2. Teste do enunciado

In [2]:
data_json = '''{"manufacturer": " BMW ", "series_name": " M4 ", "type_": "Convertible",
                "manufactured_date": "2023-01-01", "base_msrp_usd": 93300, "vin": " 1234567890 "}'''
data_json_expected_serialization = {
    'manufacturer': 'BMW', 'series_name': 'M4', 'type_': AutomobileType.convertible, 'is_electric': False,
    'manufactured_date': date(2023, 1, 1), 'base_msrp_usd': 93300.0, 'vin': '1234567890',
    'number_of_doors': 4, 'registration_country': None, 'license_plate': None,
}

car = Automobile.model_validate_json(data_json)
assert car.model_dump() == data_json_expected_serialization
print(f"car: {car!r}")
print(f"car.model_dump_json(): {car.model_dump_json()}")
print("teste do enunciado: ok")

car: Automobile(manufacturer='BMW', series_name='M4', type_=<AutomobileType.convertible: 'Convertible'>, is_electric=False, manufactured_date=datetime.date(2023, 1, 1), base_msrp_usd=93300.0, vin='1234567890', number_of_doors=4, registration_country=None, license_plate=None)
car.model_dump_json(): {"manufacturer":"BMW","series_name":"M4","type_":"Convertible","is_electric":false,"manufactured_date":"2023-01-01","base_msrp_usd":93300.0,"vin":"1234567890","number_of_doors":4,"registration_country":null,"license_plate":null}
teste do enunciado: ok


## 3. Testes extras — um por requisito

In [3]:
import json

def error_types(fn) -> list[tuple]:
    try:
        fn()
    except ValidationError as ex:
        return [(err["loc"], err["type"]) for err in ex.errors()]
    raise AssertionError("era esperado ValidationError")

base = json.loads(data_json)

# extra="forbid"
extra = error_types(lambda: Automobile.model_validate({**base, "color": "red"}))
print(f"campo extra: {extra}")
assert extra == [(("color",), "extra_forbidden")]

# enum: aceita o valor ou o membro; rejeita valor fora da enum (inclusive "convertible" minúsculo)
assert Automobile.model_validate({**base, "type_": AutomobileType.suv}).type_ is AutomobileType.suv
print(f"type_='convertible': {error_types(lambda: Automobile.model_validate({**base, 'type_': 'convertible'}))}")

# validate_assignment + str_strip_whitespace também na atribuição
car = Automobile.model_validate(base)
car.license_plate = "  AAA-BBB  "
print(f"car.license_plate (atribuído com espaços): {car.license_plate!r}")
assert car.license_plate == "AAA-BBB"
print(f"car.number_of_doors = 'quatro': {error_types(lambda: setattr(car, 'number_of_doors', 'quatro'))}")
print(f"car.type_ = 'Bike': {error_types(lambda: setattr(car, 'type_', 'Bike'))}")

# anuláveis: aceitam None explícito (inclusive por atribuição)
car.registration_country = None
assert car.registration_country is None

# validate_default: os defaults atuais são válidos, então a classe é criada e usada normalmente;
# se alguém mudar um default para um valor inválido, o erro aparece na criação da instância:
class AutomobileBadDefault(Automobile):
    number_of_doors: int = "four"

print(f"default inválido: {error_types(lambda: AutomobileBadDefault.model_validate(base))}")
print("testes extras: ok")

campo extra: [(('color',), 'extra_forbidden')]
type_='convertible': [(('type_',), 'enum')]
car.license_plate (atribuído com espaços): 'AAA-BBB'
car.number_of_doors = 'quatro': [(('number_of_doors',), 'int_parsing')]
car.type_ = 'Bike': [(('type_',), 'enum')]
default inválido: [(('number_of_doors',), 'int_parsing')]
testes extras: ok


## 4. Revisão da sua tentativa

Sua classe passa no teste do enunciado, mas tem um problema de definição:

```python
class Automobile(BaseModel):
    ...
    type_: str                 # 1ª declaração
    ...
    license_plate: str | None = None
    type_: AutomobileType      # 2ª declaração do mesmo campo
```

Discordo de manter as duas declarações porque o corpo da classe guarda as anotações num dicionário: a segunda **sobrescreve** a primeira
silenciosamente (o campo continua na 3ª posição, agora com tipo `AutomobileType`). Funciona por acaso. O que eu faria no lugar: trocar o tipo
na linha original (`type_: AutomobileType`) e apagar a duplicata. O risco do seu caminho: quem lê vê `type_: str` primeiro e assume o tipo
errado; e se a duplicata for removida "por limpeza" deixando a primeira, a validação da enum some sem nenhum erro. Linters (`ruff F811`/
`pylint`) apontam isso. Prova de que o Python aceita sem reclamar:

In [4]:
class Duplicated(BaseModel):
    a: str
    b: int = 0
    a: AutomobileType

print(f"Duplicated.model_fields['a'].annotation: {Duplicated.model_fields['a'].annotation}")
print(f"list(Duplicated.model_fields): {list(Duplicated.model_fields)}")

Duplicated.model_fields['a'].annotation: <enum 'AutomobileType'>
list(Duplicated.model_fields): ['a', 'b']


Outro ponto: o enunciado pede testes próprios ("acrescente seus próprios testes") e a tentativa não tinha nenhum além do assert do
enunciado. A seção 3 acima cobre cada requisito com um teste.

No `10 - My_Project.py`, o bloco `if __name__ == '__main__':` com o `assert` está ótimo para rodar como script — boa prática que vale manter.

## 5. Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| `model_config` | igual | igual |
| `type_` | `AutomobileType` | `AutomobileType` |
| testes | só o do enunciado | + um teste por requisito (extra, enum, atribuição com strip, `None`, default inválido) |

A solução do instrutor é idêntica no modelo. O valor agregado aqui é provar cada configuração com um teste — em especial que
`str_strip_whitespace` também se aplica na **atribuição** quando `validate_assignment=True`.